# Model Prediksi Risiko Keterlambatan Pengiriman Logistik
**World Model for Business (Kasus Marketplace Olist)**  

---
Tujuan: Membangun sistem klasifikasi risiko keterlambatan pengiriman pada dua titik waktu operasional: **Titik T0** (saat pesanan dibayar) dan **Titik T1** (saat paket diserahkan seller ke kurir), serta mengukur efektivitas intervensi awal berdasarkan metrik Presisi dan Recall pada 10% pesanan berisiko tertinggi.

In [1]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.classification import build_classification_pipeline, evaluate_top_k

sns.set_theme(style='whitegrid', palette='muted')

# Muat fitur T0 dan T1
df_t0 = pd.read_parquet('../data/processed/features_t0.parquet')
df_t1 = pd.read_parquet('../data/processed/features_t1.parquet')
df_t0 = df_t0.dropna(subset=['is_late'])
df_t1 = df_t1.dropna(subset=['is_late'])
print(f"Dataset Fitur T0 : {df_t0.shape[0]:,} baris, {df_t0.shape[1]} kolom")
print(f"Dataset Fitur T1 : {df_t1.shape[0]:,} baris, {df_t1.shape[1]} kolom")

Dataset Fitur T0 : 83,696 baris, 13 kolom
Dataset Fitur T1 : 83,695 baris, 14 kolom


## 1. Pembagian Data Latih & Uji Berbasis Waktu (Temporal Split)
Untuk menghindari kebocoran informasi masa depan (*lookahead bias*), data dibagi secara ketat berdasarkan urutan kronologis transaksi: periode Januari 2017 hingga Maret 2018 sebagai data latih, dan April hingga Juni 2018 sebagai data uji *out-of-time*.

In [2]:
def time_split(df):
    train = df[df['order_purchase_timestamp'] < '2018-04-01']
    test = df[df['order_purchase_timestamp'] >= '2018-04-01']
    return train, test

train_t0, test_t0 = time_split(df_t0)
train_t1, test_t1 = time_split(df_t1)

print(f"Data Latih T0 (s/d Maret 2018)  : {len(train_t0):,} pesanan")
print(f"Data Uji T0 (April - Juni 2018) : {len(test_t0):,} pesanan")

Data Latih T0 (s/d Maret 2018)  : 64,053 pesanan
Data Uji T0 (April - Juni 2018) : 19,643 pesanan


## 2. Pemodelan & Evaluasi Performa (Aturan Heuristik, Regresi Logistik, Random Forest)
Karena kelas keterlambatan sangat tidak seimbang (hanya ~8% pesanan terlambat), metrik yang paling relevan untuk operasional adalah **Top-10% Precision** (seberapa akurat deteksi jika CS mengambil 10% daftar teratas) dan **Top-10% Recall** (berapa porsi total keterlambatan yang berhasil ditangkap).

In [3]:
# Fitur T0 (informasi saat checkout)
cat_cols = ['primary_category']
num_cols_t0 = [
    'total_items_value', 'total_freight_value', 'estimated_delivery_days', 
    'avg_product_weight_g', 'seller_recent_late_rate_30d'
]

X_train_t0, y_train_t0 = train_t0[cat_cols + num_cols_t0], train_t0['is_late']
X_test_t0, y_test_t0 = test_t0[cat_cols + num_cols_t0], test_t0['is_late']

# Fitur T1 = T0 + handling days seller
num_cols_t1 = num_cols_t0 + ['seller_handling_days']
X_train_t1, y_train_t1 = train_t1[cat_cols + num_cols_t1], train_t1['is_late']
X_test_t1, y_test_t1 = test_t1[cat_cols + num_cols_t1], test_t1['is_late']

results = []

# 1. Baseline Heuristik Sederhana (T0)
rule_probs_t0 = (X_test_t0['seller_recent_late_rate_30d'] * X_test_t0['estimated_delivery_days']).fillna(0)
p, r, auc_val = evaluate_top_k(y_test_t0, rule_probs_t0, k_fraction=0.10)
results.append({'Model': 'Aturan Sederhana (T0)', 'Titik': 'T0', 'Top-10% Precision': p, 'Top-10% Recall': r, 'PR-AUC': auc_val})

# 2. Logistic Regression (T0)
logreg_t0 = build_classification_pipeline(cat_cols, num_cols_t0, 'logistic')
logreg_t0.fit(X_train_t0, y_train_t0)
lr_probs_t0 = logreg_t0.predict_proba(X_test_t0)[:, 1]
p, r, auc_val = evaluate_top_k(y_test_t0, lr_probs_t0, k_fraction=0.10)
results.append({'Model': 'Logistic Regression (T0)', 'Titik': 'T0', 'Top-10% Precision': p, 'Top-10% Recall': r, 'PR-AUC': auc_val})

# 3. Random Forest (T0)
rf_t0 = build_classification_pipeline(cat_cols, num_cols_t0, 'rf')
rf_t0.fit(X_train_t0, y_train_t0)
rf_probs_t0 = rf_t0.predict_proba(X_test_t0)[:, 1]
p, r, auc_val = evaluate_top_k(y_test_t0, rf_probs_t0, k_fraction=0.10)
results.append({'Model': 'Random Forest (T0)', 'Titik': 'T0', 'Top-10% Precision': p, 'Top-10% Recall': r, 'PR-AUC': auc_val})

# 4. Random Forest (T1) — Penambahan Waktu Penanganan Seller
rf_t1 = build_classification_pipeline(cat_cols, num_cols_t1, 'rf')
rf_t1.fit(X_train_t1, y_train_t1)
rf_probs_t1 = rf_t1.predict_proba(X_test_t1)[:, 1]
p, r, auc_val = evaluate_top_k(y_test_t1, rf_probs_t1, k_fraction=0.10)
results.append({'Model': 'Random Forest (T1)', 'Titik': 'T1', 'Top-10% Precision': p, 'Top-10% Recall': r, 'PR-AUC': auc_val})

res_df = pd.DataFrame(results)
res_df['Top-10% Precision'] = (res_df['Top-10% Precision']*100).round(1).astype(str) + '%'
res_df['Top-10% Recall'] = (res_df['Top-10% Recall']*100).round(1).astype(str) + '%'
res_df['PR-AUC'] = res_df['PR-AUC'].round(3)

display(res_df.sort_values('PR-AUC', ascending=False))

,Model,Titik,Top-10% Precision,Top-10% Recall,PR-AUC
3,Random Forest (T1),T1,15.3%,30.1%,0.149
2,Random Forest (T0),T0,9.3%,18.2%,0.074
1,Logistic Regression (T0),T0,8.8%,17.3%,0.071
0,Aturan Sederhana (T0),T0,6.1%,11.9%,0.055


---
## 3. Ringkasan Eksekutif & Temuan Kunci Model

Berdasarkan hasil evaluasi pada data uji *out-of-time* (April–Juni 2018):
- **Di Titik T0 (Saat Pembelian Dikonfirmasi)**:
  - Random Forest menghasilkan **Top-10% Precision 9,3%** dan **Top-10% Recall 18,2%** (PR-AUC: 0,074), mengungguli Aturan Sederhana (Precision 6,1%, Recall 11,9%, PR-AUC 0,055) dengan peningkatan *lift* sebesar 1,5x.
- **Di Titik T1 (Saat Barang Diserahkan ke Kurir)**:
  - Akurasi melonjak tajam: Random Forest (T1) mencapai **Top-10% Precision 15,3%**, **Top-10% Recall 30,1%**, dan **PR-AUC 0,149** (lonjakan performa 2x lipat dibanding T0).

> **Rekomendasi Operasional Bisnis**: 
> 1. Informasi waktu penanganan seller (*handling days*) di titik T1 adalah sinyal prediktif terkuat. Jika seller menahan barang terlalu lama sebelum diserahkan ke kurir, risiko keterlambatan melonjak signifikan.
> 2. Dengan memprioritaskan 10% pesanan berisiko tertinggi di titik T1, tim operasional/CS dapat mengintervensi dan menyelamatkan **hampir sepertiga (30,1%) dari seluruh pesanan yang akan terlambat** di platform.